# Road Segmentation Training Notebook

This notebook trains a road segmentation model on the DeepGlobe dataset.
It is designed to run on **Google Colab with GPU** or locally on CPU/MPS.

### What this notebook does
1. Installs dependencies, clones the repo, and downloads the dataset (all automatic on Colab)
2. Loads a YAML config (or lets you override settings inline)
3. Trains the model with full best practices (AMP, augmentations, early stopping, checkpointing)
4. Plots training curves and prediction visualizations
5. Optionally saves the best model to Google Drive

### How to run on Colab
1. Open this notebook in Colab (`File > Upload notebook` or from GitHub)
2. Set runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`)
3. Set `REPO_URL` in the first cell to your private GitHub repo URL
4. Set your Kaggle credentials in the dataset cell (for downloading DeepGlobe)
5. Run all cells

---
## 1. Setup & Installation

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

# ====== REPO URL ======
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
# For private repos on Colab, use a personal access token:
# REPO_URL = "https://<TOKEN>@github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    # --- Step 1: Clone the repository ---
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        print(f"Cloned to {REPO_DIR}")
    else:
        # Pull latest changes if already cloned
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
        print(f"Repo already exists at {REPO_DIR}, pulled latest.")

    # --- Step 2: Install the package + dependencies ---
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
    # Extra deps that might not be in pyproject.toml
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "kagglehub", "pyyaml", "tqdm"],
        check=True,
    )
    print("Dependencies installed.")
else:
    REPO_DIR = None
    print("Running locally — skipping Colab-specific setup.")

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Resolve project root — works in both Colab and local environments
if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    # Local: resolve relative to notebook or project root
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Ensure our package is importable (belt-and-suspenders with pip install -e)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Contents: {[p.name for p in PROJECT_ROOT.iterdir() if not p.name.startswith('.')]}")

---
## 2. Device Detection

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU — training will be slow")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 3. Download Dataset

The DeepGlobe Road Extraction dataset is hosted on Kaggle. You need Kaggle credentials:

**On Colab:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) and click "Create New Token"
2. This downloads `kaggle.json` — upload it when prompted below, OR
3. Set the environment variables directly in the cell below

**Locally:** Run `python scripts/download_data.py` from the project root (requires `~/.kaggle/kaggle.json`).

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

if not DEEPGLOBE_DATASET_DIR.exists():
    # --- Set up Kaggle credentials ---
    if IN_COLAB:
        # Option A: Upload kaggle.json interactively
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_json.exists():
            print("Upload your kaggle.json file:")
            from google.colab import files
            uploaded = files.upload()  # Upload kaggle.json
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(list(uploaded.values())[0])
            kaggle_json.chmod(0o600)
            print("Kaggle credentials saved.")

        # Option B: Set credentials directly (uncomment and fill in):
        # os.environ["KAGGLE_USERNAME"] = "your_username"
        # os.environ["KAGGLE_KEY"] = "your_api_key"

    print("Downloading DeepGlobe dataset (this takes a few minutes)...")
    from road_segmentation.data.download import download_dataset
    download_dataset()
    print(f"Dataset saved to: {DEEPGLOBE_DATASET_DIR}")
else:
    print(f"Dataset already exists at: {DEEPGLOBE_DATASET_DIR}")

# Verify
sat_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_sat.*")))
mask_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_mask.*")))
print(f"Training: {sat_count} images, {mask_count} masks")

---
## 4. Load Configuration

Load a YAML config and optionally override settings for your environment.
Available configs in `configs/`:
- `unet_resnet34.yaml` (recommended starting point)
- `deeplabv3plus_resnet50.yaml`
- `linknet_resnet34.yaml`
- `segformer_mitb2.yaml`

In [ ]:
from road_segmentation.config import load_config

# ====== CHOOSE YOUR CONFIG ======
CONFIG_NAME = "unet_resnet34.yaml"
# CONFIG_NAME = "deeplabv3plus_resnet50.yaml"
# CONFIG_NAME = "linknet_resnet34.yaml"
# CONFIG_NAME = "segformer_mitb2.yaml"

config = load_config(PROJECT_ROOT / "configs" / CONFIG_NAME)

# ====== ENVIRONMENT OVERRIDES ======
# Uncomment and adjust these for your setup:

# Quick experiment (small subset, few epochs)
# config.data.subset_size = 500
# config.training.epochs = 10

# Colab-specific
if IN_COLAB:
    config.data.num_workers = 2
    config.training.mixed_precision = torch.cuda.is_available()

# CPU-friendly settings
if device.type == "cpu":
    config.data.subset_size = config.data.subset_size or 200
    config.training.epochs = min(config.training.epochs, 10)
    config.training.mixed_precision = False
    config.data.num_workers = 0
    config.data.batch_size = 4

# Resume from checkpoint
# config.checkpoint.resume_from = "checkpoints/.../last.pth"

print(f"Config: {CONFIG_NAME}")
print(f"Model: {config.model.arch} + {config.model.encoder_name}")
print(f"Epochs: {config.training.epochs}")
print(f"Batch size: {config.data.batch_size}")
print(f"Image size: {config.data.image_size}")
print(f"Subset: {config.data.subset_size or 'full dataset'}")
print(f"Mixed precision: {config.training.mixed_precision}")

---
## 5. Build Data Pipeline

In [ ]:
import random
import numpy as np

from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_train_transform, get_val_transform
from road_segmentation.data.dataset import create_dataloaders

# Seed everything
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

# Discover pairs and split
pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
train_pairs, val_pairs = split_pairs(
    pairs,
    val_ratio=config.data.val_split_ratio,
    seed=config.data.val_split_seed,
    subset_size=config.data.subset_size,
)

# Build transforms
train_transform = get_train_transform(
    image_size=config.data.image_size,
    aug_steps=config.augmentations.train,
    mean=config.normalization.mean,
    std=config.normalization.std,
)
val_transform = get_val_transform(
    image_size=config.data.image_size,
    mean=config.normalization.mean,
    std=config.normalization.std,
)

# Create dataloaders
train_loader, val_loader = create_dataloaders(
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    train_transform=train_transform,
    val_transform=val_transform,
    batch_size=config.data.batch_size,
    num_workers=config.data.num_workers,
    pin_memory=config.data.pin_memory and device.type == "cuda",
)

print(f"Train: {len(train_pairs)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_pairs)} samples, {len(val_loader)} batches")

# Quick sanity check
batch = next(iter(train_loader))
print(f"\nBatch shapes: image={batch['image'].shape}, mask={batch['mask'].shape}")
print(f"Image range: [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
print(f"Mask unique values: {batch['mask'].unique().tolist()}")

---
## 6. Create Model

In [ ]:
from road_segmentation.models.factory import create_model, count_parameters, freeze_encoder

model = create_model(
    arch=config.model.arch,
    encoder_name=config.model.encoder_name,
    encoder_weights=config.model.encoder_weights,
    in_channels=config.model.in_channels,
    classes=config.model.classes,
    **config.model.decoder_kwargs,
)

if config.training.freeze_encoder_epochs > 0:
    freeze_encoder(model)

params = count_parameters(model)
print(f"Architecture: {config.model.arch} + {config.model.encoder_name}")
print(f"Total parameters:     {params['total'] / 1e6:.1f}M")
print(f"Trainable parameters: {params['trainable'] / 1e6:.1f}M")
if config.training.freeze_encoder_epochs > 0:
    print(f"Encoder frozen for first {config.training.freeze_encoder_epochs} epochs")

---
## 7. Train

In [ ]:
from road_segmentation.training.losses import create_loss
from road_segmentation.training.trainer import Trainer

loss_fn = create_loss(config.loss.type, config.loss.params)
print(f"Loss: {config.loss.type}")
print(f"Optimizer: {config.optimizer.type}, LR={config.optimizer.lr}")
print(f"Scheduler: {config.scheduler.type}")
print()

trainer = Trainer(
    config=config,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    device=device,
)

best_metrics = trainer.train()

---
## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt
from road_segmentation.training.visualization import plot_training_curves

plt.style.use("ggplot")
fig = plot_training_curves(trainer.history)
plt.show()

print(f"\nBest metrics:")
for k, v in best_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

---
## 9. Prediction Visualizations

Shows input image, ground truth, prediction, and error map (TP=green, FP=red, FN=blue).

In [ ]:
from road_segmentation.training.visualization import plot_prediction_samples

# Get a batch of validation samples
eval_model = trainer.ema.module if trainer.ema else trainer.model
eval_model.eval()

val_batch = next(iter(val_loader))
images = val_batch["image"][:6].to(device)
masks_gt = val_batch["mask"][:6]

with torch.no_grad():
    logits = eval_model(images)
    probs = torch.sigmoid(logits).cpu()

fig = plot_prediction_samples(
    images=val_batch["image"][:6],
    masks_gt=masks_gt,
    masks_pred=probs,
    mean=config.normalization.mean,
    std=config.normalization.std,
)
plt.show()

---
## 10. Metrics Summary

In [ ]:
import pandas as pd

history_df = pd.DataFrame(trainer.history)
print("Training History:")
display(history_df.round(4))

print(f"\nCheckpoints saved to: {trainer.ckpt_dir}")
print(f"Logs saved to: {trainer.log_dir}")

---
## 11. Save Model (Optional)

Save the best model to Google Drive for later use.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    
    import shutil
    save_dir = Path("/content/drive/MyDrive/road_segmentation")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    best_ckpt = trainer.ckpt_dir / "best.pth"
    if best_ckpt.exists():
        dest = save_dir / f"{config.logging.experiment_name}_best.pth"
        shutil.copy(best_ckpt, dest)
        print(f"Best model saved to: {dest}")
    
    # Also save the config
    shutil.copy(trainer.log_dir / "config.yaml", save_dir / f"{config.logging.experiment_name}_config.yaml")
    print("Config saved.")
else:
    print(f"Best checkpoint: {trainer.ckpt_dir / 'best.pth'}")
    print(f"Last checkpoint: {trainer.ckpt_dir / 'last.pth'}")
    print(f"\nTo resume training later:")
    print(f"  config.checkpoint.resume_from = '{trainer.ckpt_dir / 'last.pth'}'")